In [ ]:
# %% [markdown]
# ## ARIMA Model; horizon = 30
# ## Python version 3.11+

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time # To time execution

# Data and Preprocessing
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller

# ARIMA
import pmdarima as pm # For auto_arima

# Visualization
import plotly.graph_objects as go

# Plotting Style Preferences (Optional)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
ticker = "XRP-USD"
start_date = "2017-11-09" 
end_date = "2025-01-01"

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80

# --- Walk-Forward Forecast Horizon ---
h = 30 
print(f"Setting Walk-Forward Horizon to: h = {h}")

# auto_arima parameters
ARIMA_SEASONAL_PERIOD = 7 #  Weekly seasonality (or set to 1)

# Retraining configuration
RETRAIN_FREQUENCY = 0

# %% [markdown]
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True); df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min()} to {df_full.index.max()}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# %% [markdown]
# ## 4. Data Splitting

# %%
# Split Data into initial training and test sets
n_total = len(df_full)
n_train = int(train_split_ratio * n_total)
# Adjust n_test: number of times we initiate an h-step forecast
n_test = n_total - n_train - h + 1 

if n_test <= 0:
     raise ValueError(f"Not enough data for walk-forward with h={h}. Need at least {n_train + h} total points.")

train_data_df = df_full[:n_train]
test_data_full = df_full[n_train:] 

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data Available: {len(test_data_full)} points")
print(f"Number of walk-forward steps (predictions to generate & evaluate): {n_test}")
print(f"Evaluation Period Start (target t+{h}): {test_data_full.index[h-1].strftime('%Y-%m-%d')}")
print(f"Evaluation Period End (target t+{h}): {test_data_full.index[-1].strftime('%Y-%m-%d')}")

# %% [markdown]
# ## 5. Stationarity Check (on Initial Training Data)

# %%
def check_stationarity(timeseries):
    print("\nResults of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(dftest[0:4], index=["Test Statistic", "p-value", "#Lags Used", "# Observations Used"])
    for key, value in dftest[4].items(): dfoutput[f"Critical Value ({key})"] = value
    print(dfoutput.to_string())
    if dftest[1] <= 0.05: print("=> Conclusion: Data is likely Stationary (reject H0)")
    else: print("=> Conclusion: Data is likely Non-Stationary (fail to reject H0)")

print("\n--- Stationarity Check on Initial Training Data ---")
check_stationarity(train_data_df['Close'])

# %% [markdown]
# ## 6. Initial ARIMA Model Fit (using auto_arima)

# %%
print("\n--- Fitting Initial ARIMA Model on Training Data ---")
start_time_initial_train = time.time()

arima_model = pm.auto_arima(train_data_df['Close'],
                           start_p=1, start_q=1, test='adf',
                           max_p=3, max_q=3, m=ARIMA_SEASONAL_PERIOD,
                           start_P=0, seasonal=(ARIMA_SEASONAL_PERIOD > 1),
                           d=None, D=None, trace=False,
                           error_action='ignore', suppress_warnings=True,
                           stepwise=True)

end_time_initial_train = time.time()
print(f"Initial ARIMA training finished in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
print("\n--- Initial Best Model Found ---")
print(arima_model.summary())
print(f"\nBest ARIMA Order: {arima_model.order}")
print(f"Best Seasonal Order: {arima_model.seasonal_order}")

# %% [markdown]
# ## 7. Walk-Forward Validation (Rolling Forecast) Loop - t+h Steps Ahead

# %%
print(f"\n--- Starting ARIMA Walk-Forward Validation for {n_test} steps (Predicting {h} steps ahead) ---")
start_time_walk_forward = time.time()

arima_walk_forward_predictions_h_step = [] # List to store the target h-step ahead predictions

# The arima_model object will be updated iteratively

for t in range(n_test):
    # 1. Predict h steps ahead from the current state of the model
    yhat_h_steps = arima_model.predict(n_periods=h)

    # 2. Store only the prediction for the target step 'h'
    prediction_target_h = yhat_h_steps[h-1]
    arima_walk_forward_predictions_h_step.append(prediction_target_h)

    # 3. Get the ACTUAL value for the current step 't' (index n_train + t)
    actual_value_t = test_data_full['Close'].iloc[t]

    # 4. Update the model with the actual observation.
    try:
        arima_model.update(actual_value_t)
    except Exception as e:
        print(f"Warning: ARIMA update failed at step {t+1} with error: {e}. Model state might be stale.")

    # Log progress periodically
    if (t + 1) % 100 == 0:
        print(f"ARIMA Walk-Forward (h={h}) Step {t+1}/{n_test} complete.")

    # ---Full Refitting Point ---
    # if RETRAIN_FREQUENCY > 0 and (t + 1) % RETRAIN_FREQUENCY == 0 and (i + 1) < n_test:
    #     print(f"\n--- Refitting ARIMA at step {t+1}/{n_test} ---")
    #     current_history = df_full['Close'].iloc[:n_train + t + 1]
    #     arima_model = pm.auto_arima(current_history, ...) # Re-run auto_arima
    #     print("ARIMA Refitting complete.")
    # --- End Refitting ---


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nARIMA Walk-Forward (h={h}) finished in {total_walk_forward_time:.2f} seconds.")

# Ensure predictions list is numpy array for metrics
arima_walk_forward_predictions_h_step = np.array(arima_walk_forward_predictions_h_step)

# --- FINAL CHECK ---
print(f"Length of final predictions: {len(arima_walk_forward_predictions_h_step)}")
print(f"Number of walk-forward steps performed: {n_test}")

# %% [markdown]
# ## 8. Evaluate Walk-Forward Performance (t+h)

# %%
# Define the evaluation metrics function (reusable)
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics."""
    # Ensure inputs are 1D
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try:
        r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: r2 = np.nan

    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4%}")
    print(f"R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data, shifted by h-1 steps
# The prediction made at step i corresponds to the actual value at step i + h - 1 in the test set
y_test_actual_h_step = test_data_full['Close'].values[h-1:]

# Check lengths before evaluation
if len(y_test_actual_h_step) != len(arima_walk_forward_predictions_h_step):
     raise ValueError(f"Length mismatch after loop: Actual evaluation data ({len(y_test_actual_h_step)}) vs Predictions ({len(arima_walk_forward_predictions_h_step)})")

arima_wf_h_results = evaluate_forecast(y_test_actual_h_step, arima_walk_forward_predictions_h_step, f"ARIMA ({ticker})", horizon=h)

# %% [markdown]
# ## 9. Visualize Walk-Forward Results (t+h)

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")

prediction_dates = test_data_full.index[h-1:]

# Check length alignment before creating DataFrame
if len(prediction_dates) != len(arima_walk_forward_predictions_h_step):
     min_plot_len = min(len(prediction_dates), len(arima_walk_forward_predictions_h_step))
     print(f"Warning: Aligning plot data lengths to {min_plot_len}")
     prediction_dates = prediction_dates[:min_plot_len]
     plot_predictions = arima_walk_forward_predictions_h_step[:min_plot_len]
     plot_actuals = y_test_actual_h_step[:min_plot_len]
else:
     plot_predictions = arima_walk_forward_predictions_h_step
     plot_actuals = y_test_actual_h_step

results_df_wf = pd.DataFrame({
    'Actual': plot_actuals.flatten(),
    f'ARIMA (t+{h})': plot_predictions.flatten()
}, index=prediction_dates) 

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Test)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ARIMA (t+{h})'], mode='lines', name=f'ARIMA Walk-Forward (t+{h})', line=dict(color='red', dash='dash')))

fig.update_layout(
    title=f'ARIMA Walk-Forward (t+{h}) Forecast Comparison for {ticker}',
    xaxis_title="Date (Date being forecast)",
    yaxis_title="Price (USD)",
    legend_title="Data/Model",
    template="plotly_white"
)
fig.show()

# %% [markdown]
# ## 10. Walk-Forward Evaluation Period Summary

# %%
print(f"\n--- Walk-Forward Evaluation Summary ---")
print(f"Initial Training Data End Date: {train_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Walk-Forward Evaluation Period (Test Set Dates): {test_data_full.index.min().strftime('%Y-%m-%d')} to {test_data_full.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Performed: {n_test}")
print(f"Forecast Horizon Evaluated at each Step: h = {h}")
print(f"Evaluation Period (Target Dates): {test_data_full.index[h-1].strftime('%Y-%m-%d')} to {test_data_full.index[-1].strftime('%Y-%m-%d')}")




Setting Walk-Forward Horizon to: h = 30
--- Loading Data for XRP-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 2610 data points for XRP-USD from 2017-11-09 00:00:00 to 2024-12-31 00:00:00.

Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Data Available: 522 points
Number of walk-forward steps (predictions to generate & evaluate): 493
Evaluation Period Start (target t+30): 2023-08-27
Evaluation Period End (target t+30): 2024-12-31

--- Stationarity Check on Initial Training Data ---

Results of Dickey-Fuller Test:
Test Statistic            -3.935577
p-value                    0.001788
#Lags Used                26.000000
# Observations Used     2061.000000
Critical Value (1%)       -3.433527
Critical Value (5%)       -2.862943
Critical Value (10%)      -2.567517
=> Conclusion: Data is likely Stationary (reject H0)

--- Fitting Initial ARIMA Model on Training Data ---
